In [ ]:
import time
# autoreload
%load_ext autoreload
%autoreload 2

#from scipy import signal
#from scipy import interpolate
#from scipy import ndimage
import numpy as np
#import pycatch22 
#from sktime.transformations.panel import catch22
#import tsfresh
from tqdm import tqdm
import sys, os
import pandas as pd 
import dotenv
import random
load_dotenv = dotenv.load_dotenv('../.env')

# load local library
from timex import clustering
from timex import preprocessing

import gc
import seaborn as sns
import matplotlib.pyplot as plt

from collections import defaultdict

import datetime
from time import sleep

from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
import json



In [ ]:
AKI_PATH = os.environ['AKI_PATH_NEW']
os.chdir(AKI_PATH)

In [ ]:
AKI_PATH


In [ ]:
os.listdir('.')

In [ ]:
ts_data = pd.read_parquet(os.path.join(AKI_PATH, 'cleaned_DV_LCMM_data.parquet'))
ts_data = ts_data.rename(columns={"Time_since_index_FU_days": "Time_days"})
ts_data.ID = ts_data.ID.astype('int64')
ts_data['dataset_nr'] = ts_data['dataset_nr'].fillna(-1).astype('int64')


In [ ]:
ts_data.groupby('dataset_nr').ID.nunique(), ts_data.ID.nunique()

In [ ]:
MIN_TIME = 365 * 1 # days
MAX_TIME = 365 * 10 # days
MIN_MEAS_COUNT = 3 # measurements
INTERP_RES = 1
SMOOTHING_WINDOW = 365 # in days: 4 * INTERP_RES = 360
SMOOTHING_TYPE = 'gaussian_kernel'  # 'gaussian_kernel' or 'rolling_mean'
META_KEYS = ['ID', 'Time_days']
RAW_VAL_COL = 'eGFRcr_CKDEpi2009'
INT_VAL_COL = 'eGFR_int'
SM30_VAL_COL = 'eGFR_SW30'
SM365_VAL_COL = 'eGFR_SW365'

DS_SELECTION =  [list(range(1,K+1)) for K in range(1,9)]  # which datasets to include in the analysis
SELECTION_RES = [30, 60, 90, 180]
CLUSTER_NUMS =  [2, 4 , 8, 10, 12, 14, 16]

CLUSTERING_ALGO='gmm'
CLUSTER_KWARGS={"reg_covar": 1e-1, "covariance_type": "diag"}

ADD_TS_META = False
EXTRACTORS = ['custom', 'catch22']



In [ ]:
ts_data_df = ts_data[['ID', 'Time_days', 'eGFRcr_CKDEpi2009', 'dataset_nr']].dropna(subset=['eGFRcr_CKDEpi2009'])

In [25]:
file_dir = f"Results/{"_".join(EXTRACTORS)}"
if not os.path.exists(file_dir):
    os.makedirs(file_dir)

In [ ]:
for NUM_CLUSTERS in CLUSTER_NUMS:
    for SEL_RES in SELECTION_RES:
        for DS_SEL in DS_SELECTION:
            print(f'Running clustering for DS{DS_SEL}_C{NUM_CLUSTERS}_TR{SEL_RES}')


            Sel_IDS = ts_data[ts_data['dataset_nr'].isin(DS_SEL)].ID.unique()
            ts_data_run = ts_data_df[ts_data_df.ID.isin(Sel_IDS)]

            SETTINGS_DICT = {
                'MIN_TIME': MIN_TIME,
                'MAX_TIME': MAX_TIME,
                'MIN_MEAS_COUNT': MIN_MEAS_COUNT,
                'INTERP_RES': INTERP_RES,
                'SMOOTHING_WINDOW': SMOOTHING_WINDOW,
                'SMOOTHING_TYPE': SMOOTHING_TYPE,
                'META_KEYS': META_KEYS,
                'RAW_VAL_COL': RAW_VAL_COL,
                'INT_VAL_COL': INT_VAL_COL,
                'SM30_VAL_COL': SM30_VAL_COL,
                'SM365_VAL_COL': SM365_VAL_COL,
                'DS_SELECTION': DS_SEL,
                'SELECTION_RES': SEL_RES,
                'NUM_CLUSTERS': NUM_CLUSTERS,
                'CLUSTERING_ALGO': CLUSTERING_ALGO,
                'CLUSTER_KWARGS': CLUSTER_KWARGS,
                'ADD_TS_META': ADD_TS_META,
                'EXTRACTORS': EXTRACTORS
            }

            ts_clusterer = clustering.CrossSectionalClustering(smoothing=True, 
                                                            smoothing_type=SMOOTHING_TYPE,
                                                            smoothing_window_size=SMOOTHING_WINDOW,
                                                            n_skip=3,
                                                            interpolation=True, 
                                                            interpolation_resolution=INTERP_RES,
                                                            interpolation_keep_init=True,
                                                            analysis_resolution=SEL_RES,
                                                            min_measurements_per_id=MIN_MEAS_COUNT, 
                                                            min_time=MIN_TIME,
                                                            max_time=MAX_TIME,
                                                            clustering_algorithm=CLUSTERING_ALGO,
                                                            n_clusters=NUM_CLUSTERS, 
                                                            cluster_kwargs=CLUSTER_KWARGS,
                                                            id_column='ID', 
                                                            time_column='Time_days',
                                                            feature_columns=[RAW_VAL_COL],
                                                            imputation_method='knn',
                                                            cross_standardisation=True,
                                                            normalise_timeseries= "group",
                                                            normalisation_method="standard",
                                                            add_ts_meta=ADD_TS_META,
                                                            extractors=EXTRACTORS,
                                                            verbose=True)


            ts_clusterer.fit(ts_data_run)


            ts_label_df = pd.read_parquet(f'analysed_DV_LCMM_outcome_TR{SEL_RES}d.parquet')
            ts_label_df['ID'] = ts_label_df.ID.astype(int)
            ts_label_df.set_index('ID', inplace=True)
            ts_label_df.dropna(how='all', inplace=True)

            class_string = f'ds{''.join([str(c) for c in DS_SEL])}_M2splines_Llin_eGFR_C{NUM_CLUSTERS}_TR{SEL_RES}_Class' 
            proba_string = f'ds{''.join([str(c) for c in DS_SEL])}_M2splines_Llin_eGFR_C{NUM_CLUSTERS}_TR{SEL_RES}_max_prob'    
            ts_label_df_LCMM = ts_label_df.dropna(subset=[class_string])[[proba_string, class_string]]
            ts_label_df_LCMM[class_string] = ts_label_df_LCMM[class_string].astype('int')
            ts_label_df_LCMM = ts_label_df_LCMM.rename(columns={
                proba_string: 'LCMM_max_prob',
                class_string: 'LCMM_Class'
                })

            res_cluster = pd.DataFrame(zip(ts_clusterer.ts_cross_combined.index, ts_clusterer.predict()), columns=['ID', 'cluster'])
            res_cluster['cluster'] = res_cluster['cluster'].astype(int)

            res_cluster_proba = pd.DataFrame()
            res_cluster_proba['ID'] = ts_clusterer.ts_cross_combined.index
            res_cluster_proba[[f'cluster_prob_{i}' for i in range(NUM_CLUSTERS)]] =  ts_clusterer.predict_proba()

            res_final = ts_data_run.merge(res_cluster, how='left', left_on='ID', right_on='ID').dropna(subset='ID')
            res_final = res_final.merge(ts_label_df_LCMM, how='left', left_on='ID', right_index=True).dropna(subset=['cluster'])\
                                    .astype({'cluster': 'int'})

            res_final_ = res_final.groupby('ID')[['LCMM_Class', 'cluster']].first()
            res_final_['LCMM_Class'] = res_final_['LCMM_Class'].astype(int) - 1

            external_scores = {
                'ari': adjusted_rand_score(res_final_['cluster'], res_final_['LCMM_Class']), 
                'ami': adjusted_mutual_info_score(res_final_['cluster'], res_final_['LCMM_Class'])
            }

            internal_scores = ts_clusterer.get_scores()

            # combine all scores and timings in a single dictionary
            scores_dict = {
                'external_scores': external_scores,
                'internal_scores': internal_scores,
                'timings': ts_clusterer.timings,
                'settings': SETTINGS_DICT
            }

            # write scores to a jsonl file, appending if the file already exists
            with open('Results/clustering_scores_log.jsonl', 'a') as f:
                f.write(json.dumps(scores_dict) + '\n') 


            res_cluster_proba.to_csv(f'Results/{"_".join(EXTRACTORS)}/cluster_probs_ds{''.join([str(c) for c in DS_SEL])}_C{NUM_CLUSTERS}_TR{SEL_RES}.csv', sep=';')

2025-11-30 22:48:08,602 - timex.clustering - DEBUG - CrossSectionalClustering initialized
2025-11-30 22:48:08,602 - timex.clustering - INFO - Starting CrossSectionalClustering.fit(); TS shape (30127, 4)


Running clustering for DS[1]_C2_TR30
eGFRcr_CKDEpi2009


2025-11-30 22:48:08,602 - timex.clustering - INFO - Processing feature column: eGFRcr_CKDEpi2009
2025-11-30 22:48:08,602 - timex.clustering - INFO - Filtering completed for eGFRcr_CKDEpi2009 in 0.0030 seconds. TS: (30031, 3)
100%|██████████| 968/968 [00:00<00:00, 1005.58it/s]
2025-11-30 22:48:12,036 - timex.clustering - INFO - Interpolation completed for eGFRcr_CKDEpi2009 in 3.4233 seconds, TS: (3533200, 3)
100%|██████████| 968/968 [00:05<00:00, 185.34it/s]
2025-11-30 22:48:19,790 - timex.clustering - INFO - Smoothing completed for eGFRcr_CKDEpi2009 in 7.7607 seconds, TS: (3533200, 3)
2025-11-30 22:48:19,962 - timex.clustering - INFO - Selection after smoothing for eGFRcr_CKDEpi2009 in 7.7607 seconds, TS: (118096, 3)
2025-11-30 22:48:19,966 - timex.clustering - INFO - Selection after pruning NaNs for eGFRcr_CKDEpi2009 in 0.0024 seconds, TS: (61831, 3)
100%|██████████| 968/968 [00:01<00:00, 485.64it/s]
2025-11-30 22:48:22,005 - timex.clustering - INFO - Normalization completed for eGFRc

Processing custom features..


  0%|          | 0/968 [00:00<?, ?it/s]c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 2 is too high: all coefficients will experience boundary effects.
  warnings.warn(
  2%|▏         | 17/968 [00:00<00:06, 150.89it/s]\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\extractor.py:1027: RuntimeWarning: divide by zero encountered in log2
  entropy = np.nansum(psd_norm * np.log2(psd_norm))
\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\extractor.py:1027: RuntimeWarning: invalid value encountered in multiply
  entropy = np.nansum(psd_norm * np.log2(psd_norm))
 12%|█▏        | 113/968 [00:00<00:06, 140.83it/s]c:\Users\bes3.DS\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.12\Lib\site-packages\scipy\stats\_entropy.py:146: RuntimeWarning: divide by zero encountered in divide
  pk = 1.0*pk / xp.sum(pk, axis=axis, keepdims=Tru

Processing catch22 features..


100%|██████████| 968/968 [00:00<00:00, 1922.33it/s]
2025-11-30 22:48:29,399 - timex.clustering - INFO - Cross-sectional extraction completed for eGFRcr_CKDEpi2009 in 7.3930 seconds, TS cross: (968, 236)
2025-11-30 22:48:29,400 - timex.clustering - INFO - Joining completed for eGFRcr_CKDEpi2009 in 0.0000 seconds
2025-11-30 22:48:29,401 - timex.clustering - INFO - Replaced inf's by NaN's in 0.0033 seconds
2025-11-30 22:48:29,401 - timex.clustering - INFO - Removed 79 columns with more than 75.0% missingness in 0.0017 seconds
2025-11-30 22:48:29,401 - timex.clustering - INFO - Removed 3 columns with zero variance 0.0029 seconds
154it [00:00, 1877.33it/s]
2025-11-30 22:48:29,493 - timex.clustering - INFO - Removed 0 columns because of duplication in 0.0912 seconds
2025-11-30 22:48:29,508 - timex.clustering - INFO - Standardization completed in 0.0066 seconds
2025-11-30 22:48:29,509 - timex.clustering - INFO - Found 388 missing values, starting imputation
2025-11-30 22:48:29,509 - timex.clu

KeyError: ['ds1_M2splines_Llin_eGFR_C4_TR180_Class']

In [27]:
ts_label_df

,ds1_M2splines_Llin_eGFR_C2_TR30_max_prob,ds1_M2splines_Llin_eGFR_C2_TR30_Class,ds1_M2splines_Llin_eGFR_C4_TR30_max_prob,ds1_M2splines_Llin_eGFR_C4_TR30_Class,ds1_M2splines_Llin_eGFR_C6_TR30_max_prob,ds1_M2splines_Llin_eGFR_C6_TR30_Class,ds1_M2splines_Llin_eGFR_C8_TR30_max_prob,ds1_M2splines_Llin_eGFR_C8_TR30_Class,ds12_M2splines_Llin_eGFR_C2_TR30_max_prob,ds12_M2splines_Llin_eGFR_C2_TR30_Class,...,ds12345678_M2splines_Llin_eGFR_C2_TR30_max_prob,ds12345678_M2splines_Llin_eGFR_C2_TR30_Class,ds12345678_M2splines_Llin_eGFR_C4_TR30_max_prob,ds12345678_M2splines_Llin_eGFR_C4_TR30_Class,ds12345678_M2splines_Llin_eGFR_C6_TR30_max_prob,ds12345678_M2splines_Llin_eGFR_C6_TR30_Class,ds12345678_M2splines_Llin_eGFR_C8_TR30_max_prob,ds12345678_M2splines_Llin_eGFR_C8_TR30_Class,ds12345678_M2splines_Llin_eGFR_C10_TR30_max_prob,ds12345678_M2splines_Llin_eGFR_C10_TR30_Class
ID,,,,,,,,,,,,,,,,,,,,,
3565782,0.994598,1.0,0.999779,4.0,0.997585,2.0,0.775032,5.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
10445806,0.914177,1.0,0.976791,3.0,1.000000,5.0,1.000000,5.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
44055590,0.768118,1.0,0.998932,4.0,0.999986,2.0,0.755099,6.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
46780758,1.000000,1.0,0.999993,3.0,0.998637,3.0,0.625253,7.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
69161806,0.987400,2.0,0.549931,2.0,0.971412,2.0,0.890057,6.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18559134038,1.000000,1.0,0.999997,3.0,1.000000,5.0,0.830680,5.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
18565387758,1.000000,1.0,1.000000,3.0,1.000000,3.0,1.000000,7.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
18585860070,1.000000,2.0,1.000000,1.0,0.996285,1.0,1.000000,1.0,NaN,None,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None


In [ ]:
# # plot 10 random samples
# for s in ts_clusterer.ts_filtered.sample(n=1)['ID']:
#     tsv = ts_clusterer.ts_filtered.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_filtered.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='red', alpha=0.7)

#     tsv = ts_clusterer.ts_smoothed.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_smoothed.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='green', alpha=0.7)

#     tsv = ts_clusterer.ts_smoothed_filtered.query(f'ID=={s}')['eGFRcr_CKDEpi2009']
#     tst = ts_clusterer.ts_smoothed_filtered.query(f'ID=={s}')['Time_days']
#     plt.plot(tst, tsv, color='blue', alpha=0.7)

\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\clustering.py:661: RuntimeWarning: divide by zero encountered in log
  score_dict["mep"] = np.mean(-np.sum(probas * np.log(probas), axis=1))
\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\clustering.py:661: RuntimeWarning: invalid value encountered in multiply
  score_dict["mep"] = np.mean(-np.sum(probas * np.log(probas), axis=1))
